In [74]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import lightgbm as lgb
from lightgbm import LGBMClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
import mlflow
from mlflow import MlflowClient

In [75]:
df_full = pd.read_csv("df.csv")
df_red = pd.read_csv("df_red.csv")

In [ ]:
use_reduced = True  # flip to False for full run
df = df_red.copy() if use_reduced else df_full.copy()

In [77]:
df

,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,CC_SK_DPD_DEF_VAR,CC_NAME_CONTRACT_STATUS_Active_MEAN,CC_NAME_CONTRACT_STATUS_Approved_MEAN,CC_NAME_CONTRACT_STATUS_Completed_MEAN,CC_NAME_CONTRACT_STATUS_Demand_MEAN,CC_NAME_CONTRACT_STATUS_Refused_MEAN,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_COUNT
0,100002,1,0,0,0,0,202500.0,406597.5,24700.5,351000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0,1,0,1,0,270000.0,1293502.5,35698.5,1129500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0,0,1,0,0,67500.0,135000.0,6750.0,135000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0,1,0,0,0,135000.0,312682.5,29686.5,297000.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0
4,100007,0,0,0,0,0,121500.0,513000.0,21865.5,513000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,0,0,1,0,157500.0,254700.0,27558.0,225000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,1,0,0,0,72000.0,269550.0,12001.5,225000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,1,0,0,0,153000.0,677664.0,29979.0,585000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
307509,456254,1,1,0,0,0,171000.0,370107.0,20205.0,319500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [78]:
log_model_fn = {
    LGBMClassifier: mlflow.lightgbm.log_model,
    XGBClassifier: mlflow.xgboost.log_model,
    CatBoostClassifier: mlflow.catboost.log_model,
}

In [79]:
def delete_run_if_exists(experiment_name, run_name):
    client = MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        return
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
    )
    for run in runs:
        client.delete_run(run.info.run_id)

In [80]:
if mlflow.active_run():
    mlflow.end_run()

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
EXPERIMENT_NAME = "First trial"
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1774003960636, experiment_id='2', last_update_time=1774003960636, lifecycle_stage='active', name='First trial', tags={}, workspace='default'>

In [88]:
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]


# Fix noms de colonnes pour LightGBM
import re

X.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", col) for col in X.columns]

# on enleve les infs
X = X.replace([np.inf, -np.inf], np.nan)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=78
)

In [82]:
y_train.value_counts()

TARGET
0    226073
1     19932
Name: count, dtype: int64

In [83]:
y_test.value_counts()

TARGET
0    56609
1     4893
Name: count, dtype: int64

In [84]:
FBETA = 1.5


def run_cv(model, model_name, X, y, n_splits=5, tags=None):
    delete_run_if_exists(EXPERIMENT_NAME, model_name)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs, recalls, precisions, f1s, fbetas = [], [], [], [], []
    with mlflow.start_run(run_name=model_name, tags=tags or {}):
        mlflow.log_params(model.get_params())
        mlflow.set_tag("model_family", model_name)
        for fold, (tr, val) in enumerate(skf.split(X, y)):
            model.fit(X.iloc[tr], y.iloc[tr])
            preds_proba = model.predict_proba(X.iloc[val])[:, 1]
            preds = model.predict(X.iloc[val])
            aucs.append(roc_auc_score(y.iloc[val], preds_proba))
            recalls.append(recall_score(y.iloc[val], preds))
            precisions.append(precision_score(y.iloc[val], preds))
            f1s.append(f1_score(y.iloc[val], preds))
            fbetas.append(fbeta_score(y.iloc[val], preds, beta=FBETA))
        mlflow.log_metric("roc_auc_mean", np.mean(aucs))
        mlflow.log_metric("roc_auc_std", np.std(aucs))
        mlflow.log_metric("recall_mean", np.mean(recalls))
        mlflow.log_metric("recall_std", np.std(recalls))
        mlflow.log_metric("precision_mean", np.mean(precisions))
        mlflow.log_metric("precision_std", np.std(precisions))
        mlflow.log_metric("f1_mean", np.mean(f1s))
        mlflow.log_metric("f1_std", np.std(f1s))
        mlflow.log_metric("fbeta_mean", np.mean(fbetas))
        mlflow.log_metric("fbeta_std", np.std(fbetas))

        model.fit(X, y)
        log_model_fn[type(model)](model, name=model_name)

        print(
            f"{model_name} | AUC={np.mean(aucs):.4f} ± {np.std(aucs):.4f} | "
            f"Recall={np.mean(recalls):.4f} | Precision={np.mean(precisions):.4f} | "
            f"F1={np.mean(f1s):.4f} | Fbeta(β={FBETA})={np.mean(fbetas):.4f}"
        )

In [90]:
baseline_tags = {"dataset": "full", "phase": "baseline", "fbeta": str(FBETA)}

run_cv(
    LGBMClassifier(random_state=42, n_jobs=-1),
    "lgbm_baseline",
    X_train,
    y_train,
    tags=baseline_tags,
)
run_cv(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric="auc"),
    "xgb_baseline",
    X_train,
    y_train,
    tags=baseline_tags,
)
run_cv(
    CatBoostClassifier(random_state=42, verbose=0),
    "catboost_baseline",
    X_train,
    y_train,
    tags=baseline_tags,
)

[LightGBM] [Info] Number of positive: 15946, number of negative: 180858
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,231052 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 99619
[LightGBM] [Info] Number of data points in the train set: 196804, number of used features: 757
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,081025 -> initscore=-2,428504
[LightGBM] [Info] Start training from score -2,428504
🏃 View run lgbm_baseline at: http://127.0.0.1:5000/#/experiments/2/runs/127d5840b75c40419db0b7f70a9fa8fa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


KeyboardInterrupt: 